In [ ]:
import torch

In [ ]:
class NeuralNet():
    def __init__(self, input_features, hidden_layer1, hidden_layer2, bias, learning_rate):
        self.learning_rate = learning_rate

        self.W1 = torch.randn((input_features, hidden_layer1), dtype = float) * 0.1
        self.W2 = torch.randn((hidden_layer1, hidden_layer2), dtype = float) * 0.1
        self.W3 = torch.randn((hidden_layer2, ), dtype = float) * 0.1

        self.b1 = torch.full((hidden_layer1, ), bias, dtype = float)
        self.b2 = torch.full((hidden_layer2, ), bias, dtype = float)
        self.b3 = torch.full((1, ), bias, dtype = float)
    
    def train(self, X_train, y_train, X_val, y_val, iterations = 100):

        for iter in range(iterations):
            train_predictions, train_cache = self.forward(X_train)
            train_loss = self.loss(train_predictions, y_train)

            val_predictions, _ = self.forward(X_val)
            val_loss = self.loss(val_predictions, y_train)
            print(f"train loss: {train_loss}, val loss: {val_loss}")
            
            self.backward(X_train, y_train, train_predictions, train_cache)


    def forward(self, X_train):
        H1 = X_train @ self.W1 + self.b1
        A1 = self.relu(H1)

        H2 = A1 @ self.W2 + self.b2
        A2 = self.relu(H2)

        H3 = A2 @ self.W3 + self.b3

        cache = (H1, A1, H2, A2, H3)
        
        return H3, cache

    def backward(self, X, y, pred, cache):
        H1, A1, H2, A2, H3 = cache

        dH3 = self.loss_grad(H3, y) # shape: (N, 1)

        dW3 = A2.T @ dH3  # (hidden_layer2, N) @ (N, 1) = (hidden_layer2, 1)
        db3 = torch.sum(dH3, dim = 0) # shape: (1, )
        dA2 = dH3 @ self.W3.T # (N, 1) @ (1, hidden_layer2) = (N, hidden_layer2)

        dH2 = self.relu_grad(H2) * dA2 # (N, hidden_layer2)

        dW2 = A1.T @ dH2 # (hidden_layer1, N) @ (N, hidden_layer2) = (hidden_layer1, hidden_layer2)
        db2 = torch.sum(dH2) # (hidden_layer2)
        dA1 = dH2 @ self.W2.T # (N, hidden_layer2) @ (hidden_layer2, hidden_layer1) = (N, hidden_layer1)

        dH1 = self.relu_grad(H1) * dA1 # (N, hidden_layer1)

        dW1 = X.T @ dH1 # (input_featues, N) @ (N, hidden_layer1) = (input_features, hidden_layer1)
        db1 = torch.sum(dH1) # (hidden_layer, 1)

        self.W1 = self.update_parameters(self.W1, dW1, self.learning_rate)
        self.W2 = self.update_parameters(self.W2, dW2, self.learning_rate)
        self.W3 = self.update_parameters(self.W3, dW3, self.learning_rate)

        self.b1 = self.update_parameters(self.b1, db1, self.learning_rate)
        self.b2 = self.update_parameters(self.b2, db2, self.learning_rate)
        self.b3 = self.update_parameters(self.b3, db3, self.learning_rate)

    def loss(self, pred, y):
        N = y.shape[0]
        return torch.sum((pred - y) ** 2) / N
    
    def loss_grad(self, pred, y):
        N = y.shape[0]

        return 2 * (pred - y) / N

    def relu(self, r):
        return torch.relu(r).float()
    
    def relu_grad(self, r):
        return torch.where(r > 0, 1, 0).float()
    
    def update_parameters(self, param, grad, learning_rate, strategy = 'simple'):
        if strategy == "simple":
            return self.simple_update(param, grad, learning_rate)
    
    def simple_update(self, param, grad, learning_rate):
        new_param = param - learning_rate * grad
        return new_param




In [ ]:
# (10,000, 100) @ (100, 256) = (10,000, 256) @ (256, 512) = (10,000, 512) @ (512, 1) = (10,000, 1)